# 04. Proximity Buffers & Location Sensitivity Diagnostics

**Objective:** Evaluate daily sales residual anomalies across concentric Haversine geofence rings (`0-500m`, `500-1500m`, `1500-5000m`, `no_incident`), inspect directional corridor quadrants (`NW`, `NE`, `SW`, `SE`), and plot a storefront-centered spatial incident map.

**Production Sign-Off Gate:** Distance-based ring diagnostics require confirmed storefront coordinates (`coords_verified=True` in `data/raw/local_assumptions.json` or `BGA_COORDS_VERIFIED=true`).

**Mode Contract:** Reads `data/processed/incidents_raw.csv` when `DATA_MODE="raw"` (never `data/sample/incidents.csv`).

## 1. Environment Setup & Spatial Diagnostic Imports

**Objective:** Configure PID-isolated matplotlib environment, HTML/plot helpers, `StoreConfig`, and spatial ring functions.

**Inputs / Assumptions:** Prints active storefront coordinates and verification flag.

**How to Read the Output:** Confirm `coords_verified=True` before relying on exact `0-500m` ring cutoffs.

In [ ]:
import math
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from matplotlib.patches import Ellipse

from src.config import load_config
from src.notebook_utils import (
    configure_mpl_env,
    date_overlap_audit,
    load_incidents_for_mode,
    quadrant_summary,
    render_table,
    ring_confidence_intervals,
    show_fig,
)
from src.spatial_models import (
    distance_decay_weight,
    enrich_incidents_spatial,
    join_sales_with_incidents,
    summarize_geofence_sensitivity,
)

plt = configure_mpl_env()
DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
cfg = load_config(repo_root=REPO_ROOT)
print(f"DATA_MODE={DATA_MODE} | Store coords=({cfg.lat}, {cfg.lon}) | coords_verified={cfg.coords_verified} | radii_m={cfg.geofence_radii_m}")

## 2. Geofence Ring Sensitivity, 95% CIs & Corridor Quadrants

**Objective:** Load mode-appropriate sales and OSINT incidents, audit date overlap, compute ring-level residual means with 95% confidence intervals, and aggregate incidents by directional quadrant (`NW`, `NE`, `SW`, `SE`).

**Inputs / Assumptions:** Uses Haversine distances (`R = 6,371,000 m`) and normal-approximation 95% CIs (`1.96 * std / sqrt(n)` for `n >= 2`).

**How to Read the Output:** Compare `0-500m` and `500-1500m` residuals against `no_incident` and identify which corridor quadrant drives storefront disruptions.

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import load_daily_sales
    if not cfg.master_sales_db_path.is_file():
        raise FileNotFoundError(f"Missing raw SQLite DB: {cfg.master_sales_db_path}")
    daily_df = load_daily_sales(config=cfg)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])

incidents_df, incidents_source = load_incidents_for_mode(cfg, DATA_MODE, relevant_only=True)
overlap_tbl = date_overlap_audit(daily_df, incidents_df)
render_table(overlap_tbl, title=f"Date-Overlap Audit ({incidents_source.name})")
if int(overlap_tbl.loc[0, "incident_days_in_window"]) == 0:
    print("WARNING: Zero incident dates overlap the sales window.")

enriched_inc = enrich_incidents_spatial(
    incidents_df, store_lat=cfg.lat, store_lon=cfg.lon, radii_m=cfg.geofence_radii_m
)
ring_counts = (
    enriched_inc["geofence_ring"].value_counts().rename_axis("geofence_ring").reset_index(name="incident_count")
    if not enriched_inc.empty
    else pd.DataFrame(columns=["geofence_ring", "incident_count"])
)
render_table(ring_counts, title="Incident Counts by Geofence Ring")

joined = join_sales_with_incidents(
    sales_df=daily_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    radii_m=cfg.geofence_radii_m,
    max_distance_m=None,
)
ring_summary = ring_confidence_intervals(joined, summarize_geofence_sensitivity(joined))
render_table(
    ring_summary,
    title="Geofence Ring Sales Sensitivity (with 95% CI)",
    fmt={"mean_residual_pct": "{:.2%}", "median_residual_pct": "{:.2%}", "std_residual_pct": "{:.2%}", "ci_low": "{:.2%}", "ci_high": "{:.2%}", "p_value_vs_baseline": "{:.4f}"},
)

quad_tbl = quadrant_summary(enriched_inc, joined, store_lat=cfg.lat, store_lon=cfg.lon)
render_table(
    quad_tbl,
    title="Directional Corridor Quadrant Summary",
    fmt={"mean_residual_pct": "{:.2%}", "std_residual_pct": "{:.2%}"},
)

## 3. 1x3 Spatial Sensitivity & Incident Map

**Objective:** Render ring residual bars with 95% CIs and sample counts `n`, exponential vs. inverse decay kernels with geofence cutoffs, and a storefront-centered spatial scatter map.

**Inputs / Assumptions:** Concentric rings on Panel 3 convert meter radii to approximate degrees at `cfg.lat`; unverified coordinates trigger a prominent red warning banner.

**How to Read the Output:** Look for steeper negative residuals inside `0-500m`/`500-1500m` and check spatial clustering relative to the storefront star.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

y_vals = ring_summary["mean_residual_pct"] * 100.0
y_err = ring_summary["ci_half_width"].fillna(0.0) * 100.0
bars = axes[0].bar(ring_summary["ring"], y_vals, yerr=y_err, capsize=4, color="#2b8cbe", alpha=0.88)
for bar, n_val, err_v in zip(bars, ring_summary["n_obs"], y_err):
    h = bar.get_height()
    offset = (err_v + 0.35) if h >= 0 else -(err_v + 0.55)
    va = "bottom" if h >= 0 else "top"
    axes[0].text(bar.get_x() + bar.get_width() / 2.0, h + offset, f"n={int(n_val)}", ha="center", va=va, fontsize=8.5)
axes[0].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[0].set_title("Mean Sales Residual (%) by Ring (95% CI)")
axes[0].set_ylabel("Mean Residual (%)")
axes[0].tick_params(axis="x", rotation=20)

d_grid = np.linspace(0, 6000, 120)
axes[1].plot(d_grid, distance_decay_weight(d_grid, half_life_m=1500.0), label="Exponential (half-life 1500m)")
axes[1].plot(d_grid, distance_decay_weight(d_grid, half_life_m=1500.0, kernel="inverse"), linestyle="--", label="Inverse Distance")
for r in cfg.geofence_radii_m:
    axes[1].axvline(r, color="gray", linestyle=":", linewidth=0.9, label=f"{r}m cutoff" if r == cfg.geofence_radii_m[0] else None)
axes[1].set_title("Distance-Decay Weight Curves")
axes[1].set_xlabel("Distance from Storefront (m)")
axes[1].set_ylabel("Weight")
axes[1].legend(fontsize=8)

lat_col = "lat" if "lat" in enriched_inc.columns else "latitude"
lon_col = "lon" if "lon" in enriched_inc.columns else "longitude"
if not enriched_inc.empty and {lat_col, lon_col}.issubset(enriched_inc.columns):
    for cat, grp in enriched_inc.groupby("category"):
        axes[2].scatter(grp[lon_col], grp[lat_col], alpha=0.8, s=42, label=str(cat))
axes[2].scatter([cfg.lon], [cfg.lat], marker="*", s=200, color="red", zorder=6, label="Storefront")
cos_lat = max(math.cos(math.radians(cfg.lat)), 1e-6)
for r in cfg.geofence_radii_m:
    d_lat = r / 111_000.0
    d_lon = r / (111_000.0 * cos_lat)
    axes[2].add_patch(Ellipse((cfg.lon, cfg.lat), width=2 * d_lon, height=2 * d_lat, fill=False, linestyle="--", edgecolor="gray", linewidth=0.9))
axes[2].set_title("Storefront Spatial Map & Geofence Rings")
axes[2].set_xlabel("Longitude")
axes[2].set_ylabel("Latitude")
axes[2].legend(fontsize=7.5, loc="best")

if not cfg.coords_verified:
    fig.suptitle("UNVERIFIED APPROXIMATE COORDS — DO NOT USE FOR PRODUCTION SIGN-OFF", color="red", fontweight="bold", fontsize=11, y=1.02)
show_fig(fig)